# DECISION TREES (Implementation)

When do we stop splitting?
- When a node is 100% on a class (entropy = 0) [I'll use this]
- When splitting a node will result in the tree exceeding a maximum depth 
- Information gain from additional splits is less than the threshold
- When number of examples in a node is below a threshold 

In [1]:
import numpy as np

In [2]:
def entropy(p):
    # Note: 0 log2(0) == 0
    if p == 0 or p == 1:
        return 0
    
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

In [3]:
entropy(0.5)

np.float64(1.0)

### Splitting indices

In [4]:
X_train = np.array([
    [1, 1, 1],
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 1],
    [1, 1, 1],
    [1, 1, 0],
    [0, 0, 0],
    [1, 1, 0],
    [0, 1, 0],
    [0, 1, 0]
])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [5]:
def split_indices(X, feature_to_split):
    left_indices = []
    right_indices = []

    for i, x in enumerate(X):
        if x[feature_to_split] == 1:
            left_indices.append(i)

        else:
            right_indices.append(i)

    return left_indices, right_indices

In [6]:
# 0 => ear shape
# 1 => face_shape
# 2 => whiskers

print(split_indices(X_train, 0))

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])


### Weighted average entropy

In [7]:
def weighted_entropy(X, y, left_indices, right_indices):
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)

    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)

    w_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)

    return w_entropy

In [8]:
left_indices, right_indices = split_indices(X_train, 0) # Split on ear shape

print(weighted_entropy(X_train, y_train, left_indices, right_indices))

0.7219280948873623


### Information gain

In [9]:
def information_gain(X, y, left_indices, right_indices):
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)

    w_entropy = weighted_entropy(X, y, left_indices, right_indices)

    i_gain = h_node - w_entropy

    return i_gain

In [10]:
information_gain(X_train, y_train, left_indices, right_indices)

np.float64(0.2780719051126377)

### Recursive Tree

In [11]:
class Node:
    def __init__(self, feature=None, left=None, right=None, value=None):
        self.feature = feature # index of feature
        self.left = left
        self.right = right
        self.value = value # leaf value(0 or 1)

In [12]:
def build_tree(X, y):
    # Stopping condition: no data
    if len(y) == 0:
        return None

    # Stopping condition: pure node
    if all(y == y[0]):
        return Node(value = y[0]) # leaf node
    
    best_gain = None
    best_split = None
    best_feature_index = None
    
    # Calculate information gain for all possible features and pick the one with the highest information gain
    for i, feature_name in enumerate(features):
        left_indices, right_indices = split_indices(X, i)

        # Skip invalid splits
        if len(left_indices) == 0 or len(right_indices) == 0:
            continue
        
        i_gain = information_gain(X, y, left_indices, right_indices)

        if best_gain is None or i_gain > best_gain:
            best_gain = i_gain
            best_split = (left_indices, right_indices)
            best_feature_index =  i

    if best_gain is None or best_gain == 0:
        return Node(value=max(set(y), key=list(y).count)) # most common

    # Split data according to selected feature and create left and right branches of the tree
    left_indices, right_indices = best_split

    # Recursive splitting
    left_subtree = build_tree(X[left_indices], y[left_indices])
    right_subtree = build_tree(X[right_indices], y[right_indices])

    return Node(
        feature=features[best_feature_index], 
        left=left_subtree, 
        right=right_subtree
    )
            

In [13]:
features = ['Ear shape', 'Face shape', 'Whiskers']

node = build_tree(X_train, y_train)

### Visualization

In [14]:
map_features = {
    "Ear shape": {1: "Pointy", 0: "Floppy"},
    "Face shape": {1: "Round", 0: "Not Round"},
    "Whiskers": {1: "Present", 0: "Absent"}
}

outputs = {0: "Not cat", 1: "Cat"}

In [15]:
def print_tree(node, depth = 0):
    # Indentation
    indent = "  " * depth
    
    if node.value is not None:
        print(f'{indent}Prediction: {outputs.get(node.value)}')
        return 

    print(f'{indent}{node.feature} = {map_features[node.feature].get(1)}?')
    print_tree(node.left, depth + 1)

    print()

    print(f'{indent}{node.feature} = {map_features[node.feature].get(0)}?')
    print_tree(node.right, depth + 1)

In [16]:
print_tree(node)

Ear shape = Pointy?
  Face shape = Round?
    Prediction: Cat

  Face shape = Not Round?
    Prediction: Not cat

Ear shape = Floppy?
  Whiskers = Present?
    Prediction: Cat

  Whiskers = Absent?
    Prediction: Not cat


### Prediction

In [17]:
model = build_tree(X_train, y_train)

def predict(X, model=model):
    if model.value is not None:
        print(outputs.get(model.value))
        return 
        
    feature_index = features.index(model.feature)

    x_feature_index = X[feature_index]

    if x_feature_index == 1:
        predict(X, model = model.left)

    else:
        predict(X, model = model.right)

In [18]:
predict([0, 1, 0])

Not cat


### For the fun of it!

In [19]:
to_predict = []

for feature in features:
    user_input = input(f"Is the {feature} {map_features[feature].get(1)}?")

    while user_input.lower() not in ['yes', 'y', 'no', 'n']:
        print("Invalid user input. Please try again. Enter y or n")
        user_input = input(f"Is the {feature} {map_features[feature].get(1)}?")
        
    if user_input.lower() == "y" or user_input.lower() == "yes":
        to_predict.append(1)

    elif user_input.lower() == "n" or user_input.lower() == "no":
        to_predict.append(0)
            

print("Prediction: ", end='')
predict(to_predict)

Is the Ear shape Pointy? yes
Is the Face shape Round? yes
Is the Whiskers Present? yes


Prediction: Cat
